# CO₂ price and discount-rate scenarios

This quick analysis follows the Monte Carlo mean-NPV logic of the electricity and cement summary notebooks. It creates two figures:

1. low / medium / high CO₂-price scenarios, with the discount rate fixed at its current value;
2. low / medium / high discount-rate scenarios, with the CO₂ price fixed at its current value.

Each figure separates electricity and cement into their own panel. The model is simulated once per sector and the same sampled inputs are reused in every scenario, so only the selected scenario variable changes. Nothing is written to `src/`, `data/`, `results/`, or `figures/`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cement.cement_npv_monte_carlo import (
    DEFAULT_RETROFIT_BAU_MODE,
    simulate_cement_results,
)
from cement.cement_npv_summary_figures import CEMENT_TECHNOLOGY_LABELS
from electricity.electricity_npv_monte_carlo import simulate_electricity_results
from electricity.electricity_npv_summary_figures import ELECTRICITY_TECHNOLOGY_LABELS
from general_parameters import CARBON_PRICE_EUR_PER_T, INTEREST_RATE
from npv_finance import calculate_npv
from sensitivity_analysis import METRIC_SPECIFIC, METRIC_TOTAL


## Settings

The medium cases are read directly from the current shared model parameters. The low and high values below are explicit, editable scenario assumptions; they do not modify the stored thesis assumptions.


In [ ]:
SAMPLE_SIZE = 100_000
RANDOM_SEED = 42
RETROFIT_BAU_MODE = DEFAULT_RETROFIT_BAU_MODE

CURRENT_CO2_PRICE = CARBON_PRICE_EUR_PER_T.value
CURRENT_DISCOUNT_RATE = INTEREST_RATE.value

CO2_PRICE_SCENARIOS = {
    "Low": 40.0,
    "Medium": CURRENT_CO2_PRICE,
    "High": 120.0,
}
DISCOUNT_RATE_SCENARIOS = {
    "Low": 0.04,
    "Medium": CURRENT_DISCOUNT_RATE,
    "High": 0.12,
}

# Choose the NPV metric used in both scenario figures and summary tables.
# METRIC_TOTAL shows total project NPV in million EUR; METRIC_SPECIFIC shows
# EUR/MWh for electricity and EUR/t for cement.
NPV_METRIC = METRIC_TOTAL  # METRIC_TOTAL or METRIC_SPECIFIC

if NPV_METRIC not in (METRIC_TOTAL, METRIC_SPECIFIC):
    raise ValueError("NPV_METRIC must be METRIC_TOTAL or METRIC_SPECIFIC.")

pd.options.display.float_format = "{:,.3f}".format

scenario_settings = pd.DataFrame(
    {
        "Scenario": ["Low", "Medium", "High"],
        "CO2 price (EUR/tCO2)": list(CO2_PRICE_SCENARIOS.values()),
        "Discount rate": list(DISCOUNT_RATE_SCENARIOS.values()),
    }
)
print(f"Selected NPV metric: {NPV_METRIC}")
display(scenario_settings)


## Simulate the common Monte Carlo draws

These are the only full model simulations. All scenario cases below reuse these arrays.


In [ ]:
electricity_results = simulate_electricity_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    technologies=tuple(ELECTRICITY_TECHNOLOGY_LABELS),
)
cement_results = simulate_cement_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    technologies=tuple(CEMENT_TECHNOLOGY_LABELS),
    retrofit_bau_mode=RETROFIT_BAU_MODE,
)

print(f"Electricity technologies: {len(electricity_results)}")
print(f"Cement technologies: {len(cement_results)}")
print(f"Monte Carlo draws per technology: {SAMPLE_SIZE:,}")


## Scenario revaluation helpers

The helper removes the baseline emissions cost from annual cash flow, applies the selected CO₂ price, and discounts the resulting level annual cash flow at the selected rate. All other sampled model inputs remain unchanged.


In [ ]:
def scenario_npv(results, carbon_price, discount_rate):
    initial_capex = np.asarray(results["initial_capex_eur"], dtype=float)
    baseline_cash_flow = np.asarray(results["annual_net_cash_flow_eur"], dtype=float)
    baseline_emissions_cost = np.asarray(results["annual_emissions_cost_eur"], dtype=float)
    baseline_carbon_price = np.asarray(results["carbon_price_eur_per_t"], dtype=float)

    # Recover physical annual emissions without relying on sector-specific units.
    annual_emissions = np.divide(
        baseline_emissions_cost,
        baseline_carbon_price,
        out=np.zeros_like(baseline_emissions_cost),
        where=baseline_carbon_price != 0,
    )
    scenario_cash_flow = (
        baseline_cash_flow
        + baseline_emissions_cost
        - annual_emissions * carbon_price
    )
    lifetime_years = int(np.asarray(results["lifetime_years"])[0])
    return calculate_npv(
        initial_capex_eur=initial_capex,
        annual_net_cash_flow_eur=scenario_cash_flow,
        lifetime_years=lifetime_years,
        discount_rate=discount_rate,
    )


def display_metric(results, npv_eur, sector):
    if NPV_METRIC == METRIC_TOTAL:
        return npv_eur / 1_000_000.0

    output_column = "lifetime_output_mwh" if sector == "Electricity" else "lifetime_output_t"
    return npv_eur / np.asarray(results[output_column], dtype=float)


def build_scenario_summary(
    sector,
    results_by_technology,
    labels,
    scenario_name,
    scenario_values,
):
    rows = []
    for case, value in scenario_values.items():
        carbon_price = value if scenario_name == "CO2 price" else CURRENT_CO2_PRICE
        discount_rate = value if scenario_name == "Discount rate" else CURRENT_DISCOUNT_RATE

        for technology, results in results_by_technology.items():
            npv_eur = scenario_npv(results, carbon_price, discount_rate)
            metric_values = display_metric(results, npv_eur, sector)
            rows.append(
                {
                    "sector": sector,
                    "technology": technology,
                    "label": labels.get(technology, technology),
                    "scenario": case,
                    "scenario_value": value,
                    "mean_npv": float(np.mean(metric_values)),
                    "median_npv": float(np.median(metric_values)),
                    "p05": float(np.percentile(metric_values, 5)),
                    "p95": float(np.percentile(metric_values, 95)),
                }
            )
    return pd.DataFrame(rows)


def plot_sector_scenarios(summary, figure_title):
    scenario_order = ["Low", "Medium", "High"]
    colors = {"Low": "#6BAED6", "Medium": "#4472C4", "High": "#D95F5F"}
    fig, axes = plt.subplots(1, 2, figsize=(17, 8), dpi=150)

    for ax, sector in zip(axes, ["Electricity", "Cement"]):
        sector_data = summary[summary["sector"] == sector]
        medium_order = (
            sector_data[sector_data["scenario"] == "Medium"]
            .sort_values("mean_npv", ascending=True)["label"]
            .tolist()
        )
        y = np.arange(len(medium_order))
        bar_height = 0.22

        for offset, scenario in zip([-bar_height, 0.0, bar_height], scenario_order):
            values = (
                sector_data[sector_data["scenario"] == scenario]
                .set_index("label")
                .loc[medium_order, "mean_npv"]
            )
            ax.barh(
                y + offset,
                values,
                height=bar_height * 0.88,
                color=colors[scenario],
                label=scenario,
            )

        ax.set_yticks(y)
        ax.set_yticklabels(medium_order, fontsize=9)
        ax.tick_params(axis="y", left=False, length=0)
        ax.axvline(0, color="#777777", linewidth=0.9)
        ax.grid(axis="x", color="#e6e6e6", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.set_title(sector, fontsize=13)
        axis_unit = "million EUR" if NPV_METRIC == METRIC_TOTAL else ("EUR/MWh" if sector == "Electricity" else "EUR/t")
        ax.set_xlabel(f"Monte Carlo mean NPV ({axis_unit})")
        for spine in ax.spines.values():
            spine.set_visible(False)

    handles, legend_labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        legend_labels,
        title="Scenario",
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.96),
        ncol=3,
    )
    fig.suptitle(figure_title, fontsize=15, y=1.01)
    fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.90))
    return fig


## Plot 1 — CO₂-price scenarios

The discount rate stays at the current medium value while the CO₂ price changes.


In [ ]:
co2_summary = pd.concat(
    [
        build_scenario_summary(
            "Electricity",
            electricity_results,
            ELECTRICITY_TECHNOLOGY_LABELS,
            "CO2 price",
            CO2_PRICE_SCENARIOS,
        ),
        build_scenario_summary(
            "Cement",
            cement_results,
            CEMENT_TECHNOLOGY_LABELS,
            "CO2 price",
            CO2_PRICE_SCENARIOS,
        ),
    ],
    ignore_index=True,
)

display(co2_summary)
plot_sector_scenarios(
    co2_summary,
    f"NPV by CO₂-price scenario (discount rate fixed at {CURRENT_DISCOUNT_RATE:.0%})",
)
plt.show()


## Plot 2 — discount-rate scenarios

The CO₂ price stays at the current medium value while the discount rate changes.


In [ ]:
discount_summary = pd.concat(
    [
        build_scenario_summary(
            "Electricity",
            electricity_results,
            ELECTRICITY_TECHNOLOGY_LABELS,
            "Discount rate",
            DISCOUNT_RATE_SCENARIOS,
        ),
        build_scenario_summary(
            "Cement",
            cement_results,
            CEMENT_TECHNOLOGY_LABELS,
            "Discount rate",
            DISCOUNT_RATE_SCENARIOS,
        ),
    ],
    ignore_index=True,
)

display(discount_summary)
plot_sector_scenarios(
    discount_summary,
    f"NPV by discount-rate scenario (CO₂ price fixed at {CURRENT_CO2_PRICE:.0f} EUR/tCO₂)",
)
plt.show()


## Interpretation guardrails

- Low and high scenario values are sensitivity assumptions, not new thesis parameters.
- The medium cases always come from `src/general_parameters.py`.
- Scenario comparisons use common random numbers: each technology keeps the same sampled technical and market inputs across cases.
- The bars show Monte Carlo means. The summary tables also retain medians and 5th/95th percentiles for follow-up analysis.
